<a href="https://colab.research.google.com/github/Nurdaylight/Study/blob/main/Colab_LLama_try.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code from https://github.com/dmbeaglehole/neural_controllers/blob/xrfm/notebooks/harmful_shakespeare.ipynb

In [1]:
#!pip uninstall -y bitsandbytes
#!pip uninstall -y accelerate transformers
#!pip cache purge

!pip install git+https://github.com/dmbeaglehole/xRFM.git@773fae8


  Cloning https://github.com/dmbeaglehole/xRFM.git (to revision 773fae8) to /tmp/pip-req-build-a0rzmizo
  Running command git clone --filter=blob:none --quiet https://github.com/dmbeaglehole/xRFM.git /tmp/pip-req-build-a0rzmizo
  Running command git checkout -q 773fae8
  Resolved https://github.com/dmbeaglehole/xRFM.git to commit 773fae8
  Preparing metadata (setup.py) ... done


In [2]:
import sys
from pathlib import Path

In [3]:
!git clone https://github.com/dmbeaglehole/neural_controllers.git

fatal: destination path 'neural_controllers' already exists and is not an empty directory.


In [4]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/neural_controllers')

In [5]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from neural_controllers import NeuralController
from utils import harmful_dataset
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)

In [6]:
#!pip install -U bitsandbytes accelerate transformers torch

KeyboardInterrupt: 

In [9]:

    model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

    bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

    language_model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config,device_map="auto",
    dtype=torch.float16
    )
    use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=use_fast_tokenizer, padding_side="left", legacy=False)
    tokenizer.pad_token_id = 0
    model_name='llama_3_8b_it'

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [11]:
dataset = harmful_dataset(tokenizer)

README.md:   0%|          | 0.00/464 [00:00<?, ?B/s]

data/train-00000-of-00001-7008c024668c94(…):   0%|          | 0.00/14.2k [00:00<?, ?B/s]

data/test-00000-of-00001-e88521c3da18318(…):   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/128 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/384 [00:00<?, ? examples/s]

train_data 384 train_labels 384


In [ ]:
harmful_controller = NeuralController(
    language_model,
    tokenizer,
    rfm_iters=8,
    control_method='rfm',
    n_components=5
)
harmful_controller.compute_directions(dataset['train']['inputs'], np.concatenate(dataset['train']['labels']).tolist())
harmful_controller.save(concept='harmful', model_name=model_name, path='../directions/')

n_components: 5
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 2
M_batch_size         : 2048
n_components         : 5

Tuning metric: auc
Getting activations from forward passes


  6%|▌         | 19/308 [00:26<06:37,  1.38s/it]